# Контрольная работа: работа с видео

Работаем в Google Colab. Видео записываем с веб-камеры прямо здесь. **Пока идёт запись, нажимайте клавиши `s` и `t`**: ноутбук запомнит, на каком кадре какая клавиша нажата.

| | Что сделать | Балл |
|---|---|---|
| 1 | Клавиша **s** включает и выключает усиление своего цвета **на 10%** | 5 |
| 2 | Клавиша **t** включает и выключает **поворот на 180°** | 5 |

Варианты: **1 — красный, 2 — синий, 3 — зелёный.**

**Сдать в LXP:** этот ноутбук (Файл → Скачать → .ipynb) и видео `control_*.mp4`.

Можно: свои ноутбуки тем 1–4, шпаргалку ниже. Нельзя: помогать соседу и нейросети.

In [ ]:
#@title ⚙️ Запусти эту ячейку первой (помощники урока) { display-mode: "form" }
# Здесь спрятаны функции для камеры, показа видео и проверок. Читать не обязательно :)
import base64, os, shutil, subprocess, tempfile, time
import cv2
import numpy as np
import torch
from IPython.display import HTML, display

try:
    from google.colab import files, output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

FPS = 8          # частота записи с камеры (кадров в секунду)
W, H = 320, 240  # маленький размер: быстро считается и не забивает интернет

# ---------- Шрифт с кириллицей (cv2.putText русский не умеет) ----------
from PIL import Image, ImageDraw, ImageFont
import matplotlib
_FONT_DIR = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")
_fonts = {}

def put_text(frame, text, xy=(10, 10), size=24, color=(255, 255, 255), bold=True):
    """Пишет текст (можно по-русски) на кадре BGR. xy — левый верхний угол. Меняет кадр на месте и возвращает его."""
    key = (size, bold)
    if key not in _fonts:
        name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
        _fonts[key] = ImageFont.truetype(os.path.join(_FONT_DIR, name), size)
    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ImageDraw.Draw(img).text(xy, str(text), font=_fonts[key], fill=tuple(int(c) for c in color[::-1]))
    frame[:] = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    return frame

# ---------- Источники видео ----------
_REC_JS = r"""
async function recordClip(seconds, fps, w, h) {
  const box = document.createElement('div');
  box.tabIndex = 0;
  box.style = 'font:600 15px sans-serif;padding:10px;border:3px solid #1F1B2E;border-radius:14px;background:#FFF6E9;display:inline-block;outline:none;color:#1F1B2E';
  box.innerHTML = '<div id="st">Разрешите доступ к камере…</div>';
  document.querySelector('#output-area').appendChild(box);
  const st = box.querySelector('#st');
  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  const video = document.createElement('video');
  video.srcObject = stream; video.muted = true; video.playsInline = true;
  video.style = 'display:block;width:' + (w * 1.5) + 'px;border-radius:10px;margin:8px 0;transform:scaleX(-1)';
  box.appendChild(video);
  await video.play();
  const btns = document.createElement('div');
  for (const k of ['c', 'l', 'n', 's', 't']) {
    const b = document.createElement('button');
    b.textContent = k; b.style = 'margin-right:6px;padding:4px 12px;font:700 15px monospace;border:2px solid #1F1B2E;border-radius:8px;background:#fff;cursor:pointer';
    b.onclick = () => press(k);
    btns.appendChild(b);
  }
  box.appendChild(btns);
  let cur = -1; const keys = []; const frames = [];
  const shown = document.createElement('div'); box.appendChild(shown);
  function press(k) { if (cur >= 0) { keys[cur] = k.charCodeAt(0); shown.textContent = 'Нажато: ' + k + ' (кадр ' + cur + ')'; } }
  box.addEventListener('keydown', e => { if (e.key.length === 1) { press(e.key.toLowerCase()); e.preventDefault(); } });
  box.focus();
  for (let i = 3; i > 0; i--) { st.textContent = 'Запись через ' + i + '… Нажимай клавиши (или кнопки) во время записи'; await new Promise(r => setTimeout(r, 1000)); }
  const cv = document.createElement('canvas'); cv.width = w; cv.height = h;
  const ctx = cv.getContext('2d');
  const n = Math.round(seconds * fps);
  for (let i = 0; i < n; i++) {
    cur = i; keys.push(-1);
    st.textContent = '🔴 ЗАПИСЬ ' + (seconds - i / fps).toFixed(1) + ' с';
    ctx.drawImage(video, 0, 0, w, h);
    frames.push(cv.toDataURL('image/jpeg', 0.6));
    await new Promise(r => setTimeout(r, 1000 / fps));
  }
  stream.getVideoTracks()[0].stop();
  box.remove();
  return {frames, keys};
}
"""

def record_webcam(seconds=4, fps=FPS):
    """Записывает короткий клип с веб-камеры браузера. Возвращает (кадры, клавиши)."""
    if not IN_COLAB:
        print("Камера работает только в Google Colab — беру пример.")
        return sample_video(seconds, fps)
    display(HTML("<script>" + _REC_JS + "</script>"))
    data = output.eval_js(f"recordClip({seconds}, {fps}, {W}, {H})", timeout_sec=seconds + 60)
    frames = []
    for url in data["frames"]:
        buf = np.frombuffer(base64.b64decode(url.split(",", 1)[1]), np.uint8)
        frames.append(cv2.imdecode(buf, cv2.IMREAD_COLOR))
    return frames, [int(k) for k in data["keys"]]

def sample_video(seconds=4, fps=FPS):
    """Рисует пример: домик, солнце, прыгающий синий мяч и красный квадрат. Интернет не нужен."""
    frames = []
    yy = np.linspace(0, 1, H)[:, None, None]
    sky = (np.array([255, 211, 159]) * (1 - yy) + np.array([255, 246, 232]) * yy).astype(np.uint8)
    base = np.repeat(sky, W, axis=1)
    cv2.circle(base, (int(W * 0.85), int(H * 0.2)), 22, (60, 201, 255), -1)
    cv2.ellipse(base, (int(W * 0.25), H), (int(W * 0.45), int(H * 0.35)), 0, 0, 360, (126, 196, 61), -1)
    cv2.ellipse(base, (int(W * 0.85), H), (int(W * 0.4), int(H * 0.28)), 0, 0, 360, (109, 179, 47), -1)
    cv2.rectangle(base, (25, 132), (77, 185), (177, 143, 255), -1)
    cv2.fillPoly(base, [np.array([[19, 134], [51, 100], [83, 134]])], (107, 107, 255))
    n = int(seconds * fps)
    for i in range(n):
        t = i / fps
        f = base.copy()
        u = (t % 3) / 3
        x = int(W * 0.15 + W * 0.7 * (u * 2 if u < 0.5 else 2 - u * 2))
        y = int(H * 0.72 - abs(np.sin(t * np.pi * 1.3)) * H * 0.45)
        cv2.rectangle(f, (int(W * 0.6 + 40 * np.sin(t)), 150), (int(W * 0.6 + 40 * np.sin(t)) + 30, 180), (60, 60, 230), -1)
        cv2.circle(f, (x, y), 20, (255, 150, 77), -1)
        cv2.circle(f, (x, y), 20, (46, 27, 31), 2)
        frames.append(f)
    return frames, [-1] * n

def upload_video(max_frames=48):
    """Загрузка своего видеофайла (короткого!). Кадры уменьшаются до 320x240."""
    if not IN_COLAB:
        return sample_video()
    up = files.upload()
    path = next(iter(up))
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(f, (W, H)))
    cap.release()
    print(f"Прочитано кадров: {len(frames)}")
    return frames, [-1] * len(frames)

def get_video(source="camera", seconds=4):
    """source: "camera" — веб-камера, "sample" — готовый пример, "upload" — свой файл."""
    if source == "camera":
        try:
            return record_webcam(seconds)
        except Exception as e:
            print("Камера не сработала:", e, "\nБерём пример.")
            return sample_video(seconds)
    if source == "upload":
        return upload_video()
    return sample_video(seconds)

def fake_keys(n, presses):
    """Имитация нажатий: fake_keys(32, {"c": [5, 20]}) — клавиша c на кадрах 5 и 20."""
    keys = [-1] * n
    for k, idxs in presses.items():
        for i in idxs:
            if 0 <= i < n:
                keys[i] = ord(k)
    return keys

# ---------- Показ ----------
def _jpg(frame, q=80):
    if isinstance(frame, torch.Tensor):
        frame = to_frame(frame)
    frame = np.asarray(frame)
    if frame.dtype != np.uint8:
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    return base64.b64encode(cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, q])[1]).decode()

def show(*frames, titles=None, width=None):
    """Показывает один или несколько кадров рядом. Понимает numpy (H, W, 3) и тензоры (3, H, W)."""
    titles = titles or [""] * len(frames)
    cells = "".join(
        f'<figure style="margin:0 10px 0 0;display:inline-block;text-align:center">'
        f'<img src="data:image/jpeg;base64,{_jpg(f)}" style="width:{width or 320}px;border-radius:8px;border:2px solid #1F1B2E">'
        f'<figcaption style="font:600 14px sans-serif">{t}</figcaption></figure>'
        for f, t in zip(frames, titles))
    display(HTML(cells))

def to_frame(x):
    """Тензор (3, H, W) или (H, W) обратно в картинку numpy (H, W, 3) uint8."""
    x = x.detach().cpu().float()
    if x.dim() == 3 and x.shape[0] in (1, 3):
        x = x.permute(1, 2, 0)
    x = x.clamp(0, 255).byte().numpy()
    if x.ndim == 2 or x.shape[2] == 1:
        x = cv2.cvtColor(x.reshape(x.shape[0], x.shape[1]), cv2.COLOR_GRAY2BGR)
    return np.ascontiguousarray(x)

def _encode_mp4(frames, fps):
    if not shutil.which("ffmpeg"):
        return None
    h, w = frames[0].shape[:2]
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "28", "-preset", "veryfast", "-movflags", "+faststart", "-f", "mp4", "-"]
    data = b"".join(np.ascontiguousarray(f, dtype=np.uint8).tobytes() for f in frames)
    res = subprocess.run(cmd, input=data, capture_output=True)
    return res.stdout if res.returncode == 0 and res.stdout else None

def _prep(frames):
    out = [to_frame(f) if isinstance(f, torch.Tensor) else np.asarray(f) for f in frames]
    h, w = out[0].shape[:2]
    return [cv2.resize(np.clip(f, 0, 255).astype(np.uint8), (w - w % 2, h - h % 2)) for f in out]

def show_video(frames, fps=FPS, width=480):
    """Показывает список кадров как видео прямо в ноутбуке."""
    if isinstance(frames, torch.Tensor):
        frames = list(frames)
    frames = _prep(frames)
    mp4 = _encode_mp4(frames, fps)
    if mp4:
        b64 = base64.b64encode(mp4).decode()
        display(HTML(f'<video width="{width}" controls autoplay loop muted playsinline style="border-radius:10px;border:2px solid #1F1B2E">'
                     f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    else:  # нет ffmpeg — показываем GIF
        imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames]
        path = os.path.join(tempfile.gettempdir(), "preview.gif")
        imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=int(1000 / fps), loop=0)
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        display(HTML(f'<img src="data:image/gif;base64,{b64}" width="{width}" style="border-radius:10px;border:2px solid #1F1B2E">'))

def save_video(frames, path="my_video.mp4", fps=FPS, download=True):
    """Сохраняет кадры в .mp4 и (в Colab) скачивает файл."""
    frames = _prep(list(frames))
    mp4 = _encode_mp4(frames, fps)
    if mp4:
        open(path, "wb").write(mp4)
    else:
        h, w = frames[0].shape[:2]
        wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
        for f in frames:
            wr.write(f)
        wr.release()
    print(f"Сохранено: {path} ({os.path.getsize(path) // 1024} КБ)")
    if download and IN_COLAB:
        files.download(path)

# ---------- Конвейер ----------
def run_pipeline(frames, keys, process, state):
    """Тот же цикл, что и с окном OpenCV: для каждого кадра вызываем process(frame, key, state)."""
    result, t0 = [], time.perf_counter()
    for frame, key in zip(frames, keys):
        out = process(frame.copy(), key, state)
        if out is None:
            raise ValueError("process() ничего не вернул — допиши в конце  return frame")
        result.append(out)
    ms = (time.perf_counter() - t0) / max(1, len(frames)) * 1000
    print(f"Обработано кадров: {len(result)} · {ms:.1f} мс на кадр")
    return result

# ---------- Цвет и маски (пара 2) ----------
def sample_ball_center(i, fps=FPS):
    """Где на самом деле мяч на i-м кадре примера sample_video (для самопроверки)."""
    t = i / fps
    u = (t % 3) / 3
    x = int(W * 0.15 + W * 0.7 * (u * 2 if u < 0.5 else 2 - u * 2))
    y = int(H * 0.72 - abs(np.sin(t * np.pi * 1.3)) * H * 0.45)
    return x, y

def show_grid(frame, step=40):
    """Показывает кадр с сеткой координат, чтобы найти (x, y) нужного места."""
    g = frame.copy()
    for x in range(0, g.shape[1], step):
        cv2.line(g, (x, 0), (x, g.shape[0]), (255, 255, 255), 1)
        cv2.putText(g, str(x), (x + 2, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 2)
        cv2.putText(g, str(x), (x + 2, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
    for y in range(0, g.shape[0], step):
        cv2.line(g, (0, y), (g.shape[1], y), (255, 255, 255), 1)
        cv2.putText(g, str(y), (2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 0), 2)
        cv2.putText(g, str(y), (2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)
    show(g, titles=["x — по горизонтали, y — по вертикали"], width=480)

MASK = {"lower": (0, 0, 0), "upper": (179, 255, 255)}   # сюда tune_mask сохраняет выбранные границы

def tune_mask(frame):
    """Ползунки для подбора границ HSV. Выбранные значения попадают в MASK["lower"] и MASK["upper"]."""
    try:
        import ipywidgets as wd
    except ImportError:
        print("ipywidgets недоступен — впишите границы в MASK вручную")
        return
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lo, up = MASK["lower"], MASK["upper"]
    sl = lambda d, v, mx: wd.IntSlider(value=v, min=0, max=mx, description=d, continuous_update=False, layout=wd.Layout(width="420px"))
    ctl = dict(h1=sl("H от", lo[0], 179), h2=sl("H до", up[0], 179), s1=sl("S от", lo[1], 255),
               s2=sl("S до", up[1], 255), v1=sl("V от", lo[2], 255), v2=sl("V до", up[2], 255))
    def upd(h1, h2, s1, s2, v1, v2):
        MASK["lower"], MASK["upper"] = (h1, s1, v1), (h2, s2, v2)
        m = cv2.inRange(hsv, np.array(MASK["lower"]), np.array(MASK["upper"]))
        show(frame, m, cv2.bitwise_and(frame, frame, mask=m), titles=["Кадр", "Маска", "Что осталось"], width=260)
        print(f'MASK = {{"lower": {MASK["lower"]}, "upper": {MASK["upper"]}}}   # белого: {100 * (m > 0).mean():.1f}% кадра')
    display(wd.interactive(upd, **ctl))

def iou(a, b):
    """Насколько совпадают две маски: 1 — идеально, 0 — совсем не совпадают."""
    a, b = a > 0, b > 0
    return (a & b).sum() / max(1, (a | b).sum())

# ---------- Проверки ----------
def ok(msg):
    display(HTML(f'<div style="font:700 16px sans-serif;color:#1d8a52;margin:6px 0">✅ {msg}</div>'))

def fail(msg, hint=""):
    h = f'<div style="font:500 14px sans-serif;color:#6B6680;margin-top:4px">💡 {hint}</div>' if hint else ""
    display(HTML(f'<div style="font:700 16px sans-serif;color:#d93a3f;margin:6px 0">❌ {msg}{h}</div>'))

def check(cond, good, bad, hint=""):
    """Универсальная проверка: check(условие, "текст если верно", "текст если нет", "подсказка")."""
    try:
        result = bool(cond() if callable(cond) else cond)
    except Exception as e:
        result, bad = False, f"{bad} (ошибка: {e})"
    (ok(good) if result else fail(bad, hint))
    return result

print("✅ Помощники загружены. OpenCV", cv2.__version__, "· PyTorch", torch.__version__, "· Colab" if IN_COLAB else "· локально")

## Шаг 1. Кто вы и какой вариант

In [ ]:
SURNAME = "Иванов"   # ← своя фамилия латиницей или кириллицей
VARIANT = 1          # ← 1 — красный, 2 — синий, 3 — зелёный

## Шаг 2. Записываем видео

Запустите ячейку, разрешите камеру. Запись идёт 8 секунд. **Кликните мышкой по картинке с камеры** и нажимайте:
`s` → подождать секунду → `t` → подождать → `s` → подождать → `t`.

Если камеры нет — поставьте `source = "sample"`, клавиши тогда подставятся сами.

In [ ]:
source = "camera"    # "camera" или "sample"

frames, keys = get_video(source, seconds=8)
if source == "sample":
    keys = fake_keys(len(frames), {"s": [10, 40], "t": [25, 55]})

pressed = [(i, chr(k)) for i, k in enumerate(keys) if k > 0]
print("Кадров:", len(frames), "| нажатия (кадр, клавиша):", pressed)
show_video(frames)

## Шпаргалка

| Что | Код |
|---|---|
| порядок каналов в OpenCV | **B, G, R** → синий `[:, :, 0]`, зелёный `[:, :, 1]`, красный `[:, :, 2]` |
| +10% без переполнения | `np.clip(канал.astype(np.float32) * 1.1, 0, 255).astype(np.uint8)` |
| поворот на 180° | `cv2.rotate(frame, cv2.ROTATE_180)` |
| сравнить клавишу с буквой | `key == ord("s")` |
| переключатель | `state["boost"] = not state["boost"]` |

> ⚠️ Почему не просто `frame * 1.1`: в `uint8` после 255 идёт 0, и яркие места станут тёмными.

## Шаг 3. Пишем обработку кадра

Функция `process` вызывается **для каждого кадра по очереди**. `key` — код клавиши, нажатой на этом кадре (или `-1`). В `state` хранятся переключатели между кадрами.

In [ ]:
CHANNEL = {1: 2, 2: 0, 3: 1}[VARIANT]   # номер канала своего цвета в BGR

def process(frame, key, state):
    # Задание 1: клавиша s переключает state["boost"]
    # TODO

    # Задание 2: клавиша t переключает state["rotate"]
    # TODO

    if state["boost"]:
        # Задание 1: усилить канал CHANNEL на 10% без переполнения
        pass  # TODO

    if state["rotate"]:
        # Задание 2: повернуть кадр на 180°
        pass  # TODO

    return frame

## Шаг 4. Самопроверка

Ячейка прогоняет `process` на тестовых кадрах. Нужно два ✅.

In [ ]:
def _test(make_keys):
    rng = np.random.default_rng(0)
    test = [rng.integers(0, 256, (24, 32, 3), dtype=np.uint8) for _ in range(8)]
    st = {"boost": False, "rotate": False}
    out = [process(f.copy(), k, st) for f, k in zip(test, make_keys)]
    return test, out

def _boost(f):
    g = f.copy()
    g[:, :, CHANNEL] = np.clip(f[:, :, CHANNEL].astype(np.float32) * 1.1, 0, 255).astype(np.uint8)
    return g

test, out = _test(fake_keys(8, {"s": [2, 5]}))
want = [_boost(f) if 2 <= i < 5 else f for i, f in enumerate(test)]
check(lambda: all(np.abs(o.astype(int) - w.astype(int)).max() <= 1 for o, w in zip(out, want)),
      "Задание 1: s включает и выключает усиление своего канала",
      "Задание 1 пока не работает",
      "проверьте номер канала (BGR!), clip и переключатель state['boost']")

test, out = _test(fake_keys(8, {"t": [3, 6]}))
want = [cv2.rotate(f, cv2.ROTATE_180) if 3 <= i < 6 else f for i, f in enumerate(test)]
check(lambda: all(np.array_equal(o, w) for o, w in zip(out, want)),
      "Задание 2: t включает и выключает поворот",
      "Задание 2 пока не работает",
      "нужен поворот на 180° и переключатель state['rotate']")

## Шаг 5. Обрабатываем своё видео и сохраняем

In [ ]:
state = {"boost": False, "rotate": False}
result = run_pipeline(frames, keys, process, state)
show_video([np.hstack([a, b]) for a, b in zip(frames, result)], width=640)
save_video(result, f"control_{SURNAME}_v{VARIANT}.mp4")

Слева — как было, справа — после обработки. Проверьте, что цвет и поворот включаются там, где вы нажимали клавиши.

**Сдайте в LXP** ноутбук `.ipynb` и скачанное видео.